In [31]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time

def scrape_google_repositories():
    # --- 1. データベースの準備 ---
    db_name = 'test.db'
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # テーブルをリセット
    cursor.execute('DROP TABLE IF EXISTS repositories')
    cursor.execute('''
        CREATE TABLE repositories (
            name TEXT,
            language TEXT,
            stars TEXT
        )
    ''')
    conn.commit()

    # --- 2. スクレイピング設定 ---
    base_url = "https://github.com/orgs/google/repositories"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    print(f"スクレイピングを開始します: {base_url}")

    page_num = 1
    total_repos_saved = 0
    
    try:
        while True:
            current_url = f"{base_url}?page={page_num}"
            print(f"--- ページ {page_num} を処理中... ---")
            
            try:
                response = requests.get(current_url, headers=headers, timeout=10)
                if response.status_code != 200:
                     print(f"ページ取得失敗: Status {response.status_code}")
                     break
            except requests.exceptions.RequestException as e:
                print(f"アクセスエラー: {e}")
                break

            soup = BeautifulSoup(response.text, 'html.parser')
            
            # --- 3. リポジトリリストの抽出 ---
            all_lis = soup.find_all('li')
            repo_list = [li for li in all_lis if li.find('h3') and li.find('h3').find('a')]

            # div構造のバックアップ
            if not repo_list:
                all_divs = soup.find_all('div')
                repo_list = [div for div in all_divs if div.find('h3') and ('Box-row' in div.get('class', []) or 'col-12' in div.get('class', []))]

            if not repo_list:
                print("-> このページにはリポジトリがありません。全取得完了とみなします。")
                break

            print(f"  -> {len(repo_list)} 件のデータを検出")

            data_to_insert = []

            for repo in repo_list:
                try:
                    # A. リポジトリ名
                    name_tag = repo.find('h3').find('a')
                    repo_name = name_tag.text.strip()

                    # B. スターの数 (これを基準点にする)
                    star_tag = repo.find('a', href=lambda x: x and x.endswith('/stargazers'))
                    if star_tag:
                        stars = star_tag.text.strip().replace(',', '')
                    else:
                        stars = "0"

                    # C. 主要な言語 (スタータグの親要素から逆算する)
                    language = "None"
                    
                    if star_tag:
                        # スターリンクが含まれている親のdiv (メタデータ行) を取得
                        meta_row = star_tag.find_parent('div')
                        
                        if meta_row:
                            # テキストを "|" 区切りなどで取得してみる
                            # 構造例: "Java | • | Apache License | • | 327 | ..."
                            # 言語がある場合、最初の要素に来ることが多い
                            
                            # 1. itempropで再挑戦 (念のため)
                            lang_span = meta_row.find('span', itemprop='programmingLanguage')
                            if lang_span:
                                language = lang_span.text.strip()
                            else:
                                # 2. テキスト解析によるフォールバック
                                # 行内の全テキストを取得し、最初の要素を確認
                                # spanタグなどを含めてリスト化
                                items = list(meta_row.stripped_strings)
                                
                                if items:
                                    candidate = items[0] # 最初の要素
                                    
                                    # 除外キーワード判定
                                    # ライセンス名、Updated、数字(スター数) で始まっていなければ言語とみなす
                                    ignore_list = ['Updated', 'license', 'MIT', 'Apache', 'GPL', 'BSD', 'CC0']
                                    
                                    # 数字だけ、または除外リストに含まれる場合は「言語なし」と判断
                                    if (candidate.isdigit() or 
                                        any(ignored in candidate for ignored in ignore_list) or
                                        candidate == '•'):
                                         language = "None"
                                    else:
                                         language = candidate

                    data_to_insert.append((repo_name, language, stars))
                    
                except Exception:
                    continue

            # データベースに保存
            if data_to_insert:
                cursor.executemany("INSERT INTO repositories VALUES (?, ?, ?)", data_to_insert)
                conn.commit()
                total_repos_saved += len(data_to_insert)
                print(f"  -> 保存完了 (累計: {total_repos_saved}件)")
            
            page_num += 1
            time.sleep(1)

    except KeyboardInterrupt:
        print("\n処理を中断しました。")
    except Exception as e:
        print(f"予期せぬエラー: {e}")
    
    finally:
        conn.close()
        print(f"処理終了。合計 {total_repos_saved} 件を {db_name} に保存しました。")

if __name__ == "__main__":
    scrape_google_repositories()

スクレイピングを開始します: https://github.com/orgs/google/repositories
--- ページ 1 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 30件)
--- ページ 2 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 60件)
--- ページ 3 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 90件)
--- ページ 4 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 120件)
--- ページ 5 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 150件)
--- ページ 6 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 180件)
--- ページ 7 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 210件)
--- ページ 8 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 240件)
--- ページ 9 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 270件)
--- ページ 10 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 300件)
--- ページ 11 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 330件)
--- ページ 12 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 360件)
--- ページ 13 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 390件)
--- ページ 14 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 420件)
--- ページ 15 を処理中... ---
  -> 30 件のデータを検出
  -> 保存完了 (累計: 450件)
--- ページ 16 を処理中... ---
  -> 30 件のデータを検